## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
%%shell
jupyter nbconvert --to html /PATH/TO/YOUR/NOTEBOOKFILE.ipynb
/content/Erika_Yi_Full_Code_NLP_RAG_Project_Notebook.ipynb

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 151.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 114.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.3 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.3 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.3 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.3 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [ ]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [ ]:
model_path = hf_hub_download(
    repo_id= model_name_or_path,
    filename=model_basename
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [ ]:
# llm = Llama(
#     model_path=model_path,
#     n_ctx=1024,
#     n_cores=-2
# )

#### Response

In [ ]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
response("What treatment options are available for managing hypertension?")

'\n\nHypertension, or high blood pressure, is a common condition that can increase the risk of various health problems such as heart disease, stroke, and kidney damage. The good news is that there are several effective treatment options available to help manage hypertension and reduce the risk of complications. Here are some of the most commonly used treatments:\n\n1. Lifestyle modifications: Making lifestyle changes is often the first line of defense against hypertension. This may include eating a healthy diet rich in fruits, vegetables, whole grains, and lean proteins; limiting sodium intake; getting regular physical activity'

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. Source control'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical prodecure should be followed to treat it?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but some common signs and symptoms include:\n\n1. Abdominal pain: The pain is typically located in the lower right quadrant of the abdomen and may be sharp, crampy, or dull. It may start as a mild discomfort that comes and goes, but it usually worsens over time and becomes constant and severe.\n2. Loss of appetite'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nA person who has sustained a physical injury to brain tissue, also known as a traumatic brain injury (TBI), may require various treatments depending on the severity and location of the injury. Here are some common treatments recommended for TBIs:\n\n1. Emergency care: The first priority is to ensure the person's airway is clear, they are breathing, and their heart is beating normally. In severe cases, emergency surgery may be required to remove hematomas or other obstructions.\n2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as"

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or low blood pressure. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint,"

## Question Answering using LLM with Prompt Engineering

In [ ]:
system_prompt = "If not found, I don't know"

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = system_prompt+"\n"+"What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: prefix-match hit


'\nSepsis is a life-threatening condition that can arise from an infection. The management of sepsis in a critical care unit involves a multidisciplinary approach and prompt recognition and treatment. Here are some general steps that may be taken to manage sepsis in a critical care unit:\n1. Early recognition: Recognize the signs and symptoms of sepsis early, which may include fever or hypothermia, tachycardia or bradycardia, respiratory distress, altered mental status, and lactic acidosis.\n2. Source control: Identify'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right quadrant of the abdomen and may start as a mild discomfort that gradually worsens over time. The pain may be constant or intermittent and may be aggravated by movement, deep breathing, or coughing.\n2. Loss of appetite'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nA person who has sustained a physical injury to brain tissue, also known as a traumatic brain injury (TBI), may require various treatments depending on the severity and location of the injury. Here are some common treatments recommended for TBIs:\n\n1. Emergency care: The first priority is to ensure the person's airway is clear, they are breathing, and their heart is beating normally. In severe cases, emergency surgery may be required to remove hematomas or other obstructions.\n2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as"

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input ="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or low blood pressure. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint,"

## Data Preparation for RAG

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

### Loading the Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
import os

files_and_directories = os.listdir('/content/drive/MyDrive/Colab Notebooks/')
for item in files_and_directories:
    print(item)

BigDataPart2
BigDataPart1
BigDataPart3
big-data-challenge
main1_level1_finalized.ipynb
config.py
main2_level1_finalized.ipynb
medical_diagnosis_manual.pdf
Full_Code_NLP_RAG_Project_Notebook (1).ipynb


In [ ]:
import os

# Define the base directory where the PDF is located
base_dir = '/content/drive/MyDrive/Colab Notebooks/'

# Define the specific PDF file name
pdf_file_name = 'medical_diagnosis_manual.pdf'

# Construct the full path using os.path.join
manual_pdf_path = os.path.join(base_dir, pdf_file_name)

print(f"Corrected manual PDF path: {manual_pdf_path}")

Corrected manual PDF path: /content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf


In [ ]:
# Now, let's try loading the PDF again with the corrected path
pdf_loader = PyMuPDFLoader(manual_pdf_path)
manual = pdf_loader.load()
print(f"Successfully loaded {len(manual)} pages from the PDF.")

Successfully loaded 4114 pages from the PDF.


### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
  print(f"Page Number : {i+1}",end="\n")
  print(manual[i].page_content,end="\n")

Page Number : 1
yi.erika0220@gmail.com
9UQFXT5OPG
ant for personal use by yi.erika0220@gm
shing the contents in part or full is liable
Page Number : 2
yi.erika0220@gmail.com
9UQFXT5OPG
This file is meant for personal use by yi.erika0220@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .........................................................................................................................................................................................

#### Checking the number of pages

In [ ]:
len(manual)

4114

### Data Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=128,
    separators=['\n\n', '\n', ' ', '']
)

In [ ]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
len(document_chunks)

9387

In [ ]:
document_chunks[0].page_content

'yi.erika0220@gmail.com\n9UQFXT5OPG\nant for personal use by yi.erika0220@gm\nshing the contents in part or full is liable'

In [ ]:
document_chunks[2].page_content

'Table of Contents\n1\nFront    ................................................................................................................................................................................................................\n1\nCover    .......................................................................................................................................................................................................\n2\nFront Matter    ...........................................................................................................................................................................................\n53\n1 - Nutritional Disorders    ...............................................................................................................................................................\n53\nChapter 1. Nutrition: General Considerations    ...........................................................................................

In [ ]:
document_chunks[3].page_content

"229\nChapter 19. Inflammatory Bowel Disease    .........................................................................................................................\n241\nChapter 20. Diverticular Disease    ...........................................................................................................................................\n246\nChapter 21. Anorectal Disorders    ............................................................................................................................................\n254\nChapter 22. Tumors of the GI Tract    ......................................................................................................................................\n275\n3 - Hepatic & Biliary Disorders    ............................................................................................................................................\n275\nChapter 23. Approach to the Patient With Liver Disease    .........................................

### Embedding

In [ ]:
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipython-input-3002436811.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [ ]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  384


True

In [ ]:
embedding_1,embedding_2

([-0.07799344509840012,
  0.05522589385509491,
  -0.004778371192514896,
  -0.025384459644556046,
  0.07581344246864319,
  -0.018376892432570457,
  0.12176954746246338,
  0.03152600675821304,
  0.005126860458403826,
  0.009516126476228237,
  0.08502651751041412,
  -0.0004012853023596108,
  0.006589178927242756,
  -0.013660921715199947,
  -0.02281372994184494,
  -0.024491511285305023,
  -0.0637589693069458,
  -0.06164788827300072,
  -0.088883176445961,
  -0.027152109891176224,
  -0.017307845875620842,
  -0.035725172609090805,
  -0.00319072138518095,
  -0.07279198616743088,
  -0.04981670156121254,
  0.03761101886630058,
  -0.02827776037156582,
  0.06178930774331093,
  0.008998526260256767,
  -0.06604358553886414,
  0.023974651470780373,
  0.038113612681627274,
  0.058909591287374496,
  -0.013527078554034233,
  0.03196187689900398,
  0.058886315673589706,
  -0.018196461722254753,
  -0.09924376755952835,
  0.009629708714783192,
  -0.04748279228806496,
  1.0143363397219218e-05,
  0.011993570

### Vector Database

In [ ]:
out_dir = 'medical_db'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [ ]:
vectorstore = Chroma.from_documents(
    documents=document_chunks,
    embedding=embedding_model,
    persist_directory=out_dir
    )
# vectorstore.persist()

In [ ]:
vectorstore = Chroma(
    persist_directory=out_dir,
    embedding_function=embedding_model
    )

In [ ]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
vectorstore.similarity_search("Disorder",k=3)

[Document(metadata={'creationDate': 'D:20120615054440Z', 'keywords': '', 'total_pages': 4114, 'source': '/content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf', 'format': 'PDF 1.7', 'moddate': '2025-11-13T01:12:47+00:00', 'file_path': '/content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf', 'author': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'creator': 'Atop CHM to PDF Converter', 'subject': '', 'trapped': '', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'modDate': 'D:20251113011247Z', 'page': 3261}, page_content='• Conduct disorder, characterized by poor control of aggressive impulses against others, possibly\nredirected against self (see p. 3060)\nMore immediate precipitating factors can include\n• Loss of self-esteem (eg, resulting from family arguments, a humiliating disciplinary episode, pregnancy,\nor school failure)\n• Loss of a boyfriend or girlfriend\n• Loss of f

### Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [ ]:
rel_docs = retriever.get_relevant_documents(query="What is the protocol for managing sepsis in a critical care unit?")
rel_docs

[Document(metadata={'modDate': 'D:20251113011247Z', 'creationdate': '2012-06-15T05:44:40+00:00', 'subject': '', 'trapped': '', 'keywords': '', 'source': '/content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'format': 'PDF 1.7', 'page': 2400, 'creationDate': 'D:20120615054440Z', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'file_path': '/content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creator': 'Atop CHM to PDF Converter', 'author': '', 'moddate': '2025-11-13T01:12:47+00:00'}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric

In [ ]:
model_output = llm(
    "What is the protocol for managing sepsis in a critical care unit?",
    max_tokens=128,
    temperature=0
)

Llama.generate: prefix-match hit


In [ ]:
model_output['choices'][0]['text']

'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. Source control'

### System and User Prompt Template

In [ ]:
qna_system_message = "Describe the assistant's role"

In [ ]:
qna_user_message_template = """
{context}
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input,top_k=20)

Llama.generate: prefix-match hit


'The protocol for managing sepsis in a critical care unit includes the following steps:\n1. Suspect sepsis or septic shock based on clinical signs and symptoms such as shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms (abdominal pain, nausea, vomiting, diarrhea).\n2. Obtain cultures of blood and any other appropriate specimens for laboratory analysis.\n3. Administer empiric antibiotics based on the suspected organism or microbiological culture results.\n4. Monitor vital signs closely and provide supportive care'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input,top_k=20)

Llama.generate: prefix-match hit


"The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing"

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input,top_k=20)

NameError: name 'retriever' is not defined

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input,top_k=20)

Llama.generate: prefix-match hit


"The assistant's role in this scenario would be to provide support and assistance to the person with a brain injury and their family members. This may include:\n1. Helping with daily living activities such as bathing, dressing, feeding, and mobility.\n2. Providing emotional support and counseling to help cope with the trauma and adjust to the new situation.\n3. Assisting with communication needs, such as using alternative methods of communication if speech is impaired.\n4. Coordinating care with healthcare professionals, including neurologists, rehabilitation therapists, and other specialists.\n5. Providing information"

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input,top_k=20)

Llama.generate: prefix-match hit


'A person who has fractured their leg during a hiking trip should first be assessed for any life-threatening injuries or conditions. If there are no immediate threats to their safety, the following precautions and treatment steps can be taken:\n\n1. Immobilize the affected leg: Use a splint or sling to prevent further injury and decrease pain. This will also help to prevent fat embolism in long-bone fractures.\n2. Control bleeding: Apply direct pressure to any visible wounds with a clean cloth to stop the bleeding. If the bleeding does not stop, seek medical attention immediately'

### Fine-tuning

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input,temperature=0.5)

Llama.generate: prefix-match hit


'The protocol for managing sepsis in a critical care unit includes the following steps:\n1. Suspect sepsis or septic shock based on clinical signs and symptoms, such as shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms (abdominal pain, nausea, vomiting, diarrhea).\n2. Obtain cultures of blood and any other appropriate specimens for laboratory analysis.\n3. Administer empiric antibiotics as soon as possible after appropriate cultures are obtained. The choice of antibiotics depends on the suspected source of infection'

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,temperature=0.5)

Llama.generate: prefix-match hit


'The common symptoms for appendicitis include epigastric or periumbilical pain that shifts to the right lower quadrant, anorexia, nausea, vomiting, abdominal tenderness, and a low-grade fever. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur.\n\nAppendicitis cannot be cured via medicine alone as it requires surgical removal of the appendix to prevent complications such as perforation, gangrene, and peritonitis. The treatment for acute appendicitis'

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_2 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_2,temperature=0.5)

Llama.generate: prefix-match hit


'Alopecia Areata\nSudden patchy hair loss, also known as alopecia areata, is a common condition characterized by round or oval areas of complete hair loss on the scalp, beard, or other hair-bearing areas. The underlying cause of alopecia areata is not well understood but is believed to be an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental triggers.\nThe diagnosis of alopecia areata is typically made based on the clinical presentation and exclusion of other causes of hair loss. Microscopic hair examination or'

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_2 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_2,temperature=0.5)

Llama.generate: prefix-match hit


"The assistant's role in this situation would be to provide support and assistance to the patient as they undergo medical treatment and rehabilitation. This could include helping with activities of daily living, such as bathing, dressing, and feeding, as well as providing emotional support and encouragement during the recovery process. The assistant may also help coordinate appointments with healthcare providers, manage medications, and assist with physical therapy exercises as prescribed by a healthcare professional. Additionally, the assistant could help the patient's family members understand the condition and treatment plan, as well as provide them with resources for support and education. It is important for the"

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_2 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_2,temperature=0.5)

Llama.generate: prefix-match hit


'A person who has fractured their leg during a hiking trip requires prompt medical attention. Here are some necessary precautions and treatment steps:\n\n1. Immobilize the affected limb: Use a splint or sling to immobilize the leg and prevent further injury. This will also help reduce pain.\n2. Control bleeding: Apply direct pressure to the wound with a clean cloth to control any bleeding. If bleeding does not stop, seek medical attention immediately.\n3. Pain relief: Administer pain medication as prescribed by a healthcare professional. Over-the-counter pain relievers like acet'

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message  = ""

In [ ]:
relevance_rater_system_message = ""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What is the protocol for managing sepsis in a critical care unit?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 of a biological attack in a critical care unit, strict infection control measures should be implemented to prevent the spread of the agent. This includes wearing appropriate personal protective equipment (PPE), isolating affected patients, and decontaminating the environment. In addition to standard sepsis management, specific treatments for the particular biological agent may be required. It is important for critical care units to have a well-developed response plan in place for dealing with such situations.

 of a biological attack in a critical care unit, strict infection control measures should be implemented to prevent the spread of the agent. This includes wearing appropriate personal protective equipment (PPE), isolating affected patients, and implementing environmental decontamination procedures. In addition, specific treatments for the particular biological agent may be required, such as antibiotics for anthrax or supportive care for viral hemorrhagic fevers. It is important 

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 The common symptoms for appendicitis include epigastric or periumbilical pain that shifts to the right lower quadrant, brief nausea, vomiting, anorexia, and abdominal tenderness, particularly at McBurney's point. Fever is also common. However, these classic findings appear in less than 50% of patients, and symptoms can vary greatly, especially in infants, children, elderly patients, and pregnant women.

Appendicitis cannot be cured via medicine alone as it requires surgical removal of the appendix to prevent complications such as perforation, gangrene, and peritonitis. Treatment involves open or laparoscopic appendectomy, which should be preceded by IV antibiotics. If surgery is impossible, antibiotics can improve survival rates but are not curative. In cases where a large inflammatory mass is found involving the appendix, terminal ileum, and cecum, resection of the entire mass and ileocolostomy may be necessary.

Hernias of the abdominal wall are a different condition that requires e

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 also occur in people with no known family history of the condition, suggesting that other factors may play a role. Stress, illness, or trauma can trigger the onset of alopecia areata in genetically susceptible individuals. In some cases, vitamin deficiencies or certain medications may be contributing factors. A thorough evaluation by a healthcare professional is necessary to determine the underlying cause and develop an appropriate treatment plan.

 also occur in people with no known family history of the condition, suggesting that other factors may play a role. Stress, illness, or trauma can trigger the onset of alopecia areata in genetically susceptible individuals. In some cases, vitamin deficiencies or certain medications may be contributing factors. A thorough evaluation by a healthcare professional is necessary to determine the underlying cause and develop an appropriate treatment plan.


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 You have provided an accurate summary of the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function. The treatments include supportive care, specific treatments based on the type and cause of the brain injury, rehabilitation, medications, and long-term care. Supportive care focuses on maintaining the patient's overall health and preventing further damage to the brain. Specific treatments may involve surgical intervention for certain types of brain injuries. Rehabilitation is necessary when neurologic deficits persist, and it should be provided through a team approach that combines physical, occupational, and speech therapy, skill-building activities, and counseling. Medications may be prescribed to manage specific symptoms and conditions. Long-term care is often required for patients with persistent neurologic deficits. It's important to note that each case is unique, and treatment plans

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Additionally, it's important to consider the risk of Deep Vein Thrombosis (DVT) during the recovery process. While walking or otherwise moving the legs periodically can help reduce this risk, those at higher risk may require additional preventive treatment such as thrombosis prophylaxis with medications like low-dose unfractionated heparin (UFH), low molecular weight heparin (LMWH), warfarin, or newer anticoagulants. Consult a healthcare professional for specific recommendations based on the individual's risk factors and injury severity.

Here are some additional steps to prevent DVT:

1. Elevate the legs: Keep the legs elevated above heart level as much as possible when resting.
2. Avoid prolonged sitting or standing: Try to avoid long periods of inactivity, especially during travel or extended recovery periods.
3. Exercise regularly: Engage in regular lower extremity exercises, such as ankle pumps and leg extensions, to promote blood flow and prevent stagnation.
4. Wear compression 

## Actionable Insights and Business Recommendations

<font size=6 color='blue'>Power Ahead</font>

Understand disorders